# Notebook 19: Bayesian VOI Algorithmic Ledger Offline

This notebook tests a DDXPlus-native Bayesian value-of-information evidence ledger as the next algorithmic successor after the graph-shortlist ablations.

Core idea:

```text
visible evidence -> posterior over 49 diagnoses -> value of information for each legal root -> request or stop
```

This notebook is offline-only. It makes no API calls and does not use hidden test labels or hidden differentials inside the policy. Hidden test evidence is revealed only when the offline environment explicitly requests that root.

In [1]:
from __future__ import annotations

import ast
import json
import math
import os
import random
import time
import zipfile
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.metrics import f1_score
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

pd.set_option("display.max_colwidth", 160)
pd.set_option("display.max_rows", 50)
if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("high")


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "scripts" / "download_ddxplus.py").exists():
            return candidate
    raise FileNotFoundError("Could not locate baseline_model project root.")


ROOT = find_project_root(Path.cwd().resolve())
DATASET_ENV_VAR = "DDXPLUS_DATASET_DIR"
DEFAULT_DATASET_DIR = ROOT / "dataset"
LEGACY_DATASET_DIR = ROOT / ".data" / "ddxplus" / "22687585"
DATASET_DIR = (
    Path(os.environ[DATASET_ENV_VAR]).expanduser()
    if os.environ.get(DATASET_ENV_VAR)
    else DEFAULT_DATASET_DIR
    if DEFAULT_DATASET_DIR.exists()
    else LEGACY_DATASET_DIR
    if LEGACY_DATASET_DIR.exists()
    else DEFAULT_DATASET_DIR
)

# Set True for a quick artifact-contract smoke run; keep False for the full 49-case offline experiment.
SMOKE_MODE = False
if os.environ.get("BAYESIAN_VOI_SMOKE_MODE") == "1":
    SMOKE_MODE = True

RANDOM_SEED = 2919
BASE_RUN_NAME = "bayesian_voi_offline_notebook13_49case_v1"
RUN_NAME = BASE_RUN_NAME + ("_smoke" if SMOKE_MODE else "")
ARTIFACT_ROOT = ROOT / "artifacts" / "bayesian_voi_ledger" / RUN_NAME
FIGURE_DIR = ARTIFACT_ROOT / "figures"
CACHE_DIR = ARTIFACT_ROOT / "cache"
for directory in [ARTIFACT_ROOT, FIGURE_DIR, CACHE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

NOTEBOOK13_RUN = ROOT / "artifacts" / "sequential_hybrid_mlp_feedback" / "selected_stop_live_confirmation_49case_v1"
NOTEBOOK17_RUN = ROOT / "artifacts" / "graph_algorithmic_ledger" / "live_medkgi_graph_shortlist_pilot24_v1"
NOTEBOOK18_RUN = ROOT / "artifacts" / "graph_algorithmic_ledger" / "live_graph_advisory_hybrid_shortlist_pilot24_v1"
PARTIAL_MLP_SELECTED = ROOT / "artifacts" / "one_shot_partial_evidence" / "selected_model.json"

ALPHA = 1.0
LOG_LIKELIHOOD_CLIP = 4.0
AGE_SEX_WEIGHT = 0.25
BAYES_EVIDENCE_WEIGHT = 1.0
MLP_FUSION_WEIGHT = 0.60
BAYES_FUSION_WEIGHT = 0.40
TOP_BAYES_VOI_ROOTS = 32
MAX_COUNTERFACTUAL_OUTCOMES = 8
MAX_REQUEST_CAP = 5 if SMOKE_MODE else 24
LAMBDA_COSTS = [0.05] if SMOKE_MODE else [0.00, 0.02, 0.05, 0.10, 0.15]
SMOKE_MAX_CASES = 3
TRAIN_LIKELIHOOD_NROWS = 20_000 if SMOKE_MODE else None
CALIBRATION_ROWS = 1_000 if SMOKE_MODE else 10_000

STOP_CONFIG = {
    "fused_confidence_min": 0.70,
    "fused_margin_min": 0.20,
    "fused_entropy_max": 0.35,
    "contradiction_max": 1.50,
    "min_requests": 1,
}

UTILITY_WEIGHTS = {
    "expected_fused_entropy_reduction": 0.55,
    "expected_margin_gain": 0.20,
    "contradiction_resolution_gain": 0.15,
    "rare_recovery_bonus": 0.10,
}

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

print("Project root:", ROOT)
print("Dataset dir :", DATASET_DIR)
print("Run name    :", RUN_NAME)
print("Smoke mode  :", SMOKE_MODE)
print("Run dir     :", ARTIFACT_ROOT)

Project root: /Users/bilalawan/claw/assignments/baseline_model
Dataset dir : /Users/bilalawan/claw/assignments/baseline_model/dataset
Run name    : bayesian_voi_offline_notebook13_49case_v1
Smoke mode  : False
Run dir     : /Users/bilalawan/claw/assignments/baseline_model/artifacts/bayesian_voi_ledger/bayesian_voi_offline_notebook13_49case_v1


## 1. Dataset, Parsing, And BASD Observation Encoding

This section mirrors the existing notebook-first utilities: DDXPlus files, stable `case_id`s, evidence token parsing, and BASD-style slot encoding for the partial-evidence MLP.

In [2]:
SPLIT_TO_FILENAME = {
    "train": "release_train_patients.zip",
    "validate": "release_validate_patients.zip",
    "test": "release_test_patients.zip",
}
REQUIRED_DATASET_FILES = [
    "release_evidences.json",
    "release_conditions.json",
    "release_train_patients.zip",
    "release_validate_patients.zip",
    "release_test_patients.zip",
]
ABSENT_STATE = "__ABSENT__"
PRESENT_STATE = "__PRESENT__"
EPS = 1e-12


def require_file(path: Path, description: str) -> Path:
    if not path.exists():
        raise FileNotFoundError(f"Missing {description}: {path}")
    return path


def ensure_dataset_present(dataset_dir: Path) -> dict[str, Path]:
    missing = [name for name in REQUIRED_DATASET_FILES if not (dataset_dir / name).exists()]
    if missing:
        raise FileNotFoundError(f"Missing DDXPlus files in {dataset_dir}: {missing}. Run scripts/download_ddxplus.py first.")
    return {name: dataset_dir / name for name in REQUIRED_DATASET_FILES}


def load_json(path: Path) -> dict[str, Any]:
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)


def zip_table_member(zip_path: Path) -> str:
    with zipfile.ZipFile(zip_path, "r") as archive:
        members = [name for name in archive.namelist() if not name.endswith("/")]
        if not members:
            raise ValueError(f"Archive is empty: {zip_path}")
        return next((name for name in members if name.endswith(".csv")), members[0])


def load_patient_split(zip_path: Path, nrows: int | None = None) -> pd.DataFrame:
    member = zip_table_member(zip_path)
    with zipfile.ZipFile(zip_path, "r") as archive:
        with archive.open(member) as handle:
            return pd.read_csv(handle, nrows=nrows)


def attach_split_metadata(frame: pd.DataFrame, split: str) -> pd.DataFrame:
    frame = frame.copy()
    frame["source_row_index"] = frame.index.astype(int)
    frame["split"] = split
    frame["case_id"] = split + ":" + frame["source_row_index"].astype(str)
    return frame


def safe_parse_list(raw: Any) -> list[Any]:
    if isinstance(raw, list):
        return raw
    if raw is None:
        return []
    if isinstance(raw, float) and np.isnan(raw):
        return []
    text = str(raw).strip()
    if text == "" or text.lower() == "nan":
        return []
    try:
        parsed = ast.literal_eval(text)
    except (ValueError, SyntaxError):
        return [text]
    if isinstance(parsed, list):
        return parsed
    return [parsed]


def parse_evidence_token(token: str) -> tuple[str, str | None]:
    token = str(token)
    if "_@_" not in token:
        return token, None
    root_id, value = token.split("_@_", 1)
    return root_id, value


def encode_age(age: int) -> int:
    age = int(age)
    if age < 1:
        return 0
    if age <= 4:
        return 1
    if age <= 14:
        return 2
    if age <= 29:
        return 3
    if age <= 44:
        return 4
    if age <= 59:
        return 5
    if age <= 74:
        return 6
    return 7


def encode_sex(sex: str) -> int:
    sex = str(sex)
    if sex == "M":
        return 0
    if sex == "F":
        return 1
    raise ValueError(f"Unexpected sex value: {sex}")


@dataclass
class ObservationSchema:
    root_ids: list[str]
    slot_slices: dict[str, tuple[int, int]]
    data_types: dict[str, str]
    possible_values: dict[str, list[str]]
    default_values: dict[str, str | None]
    categorical_integer_roots: set[str]
    question_text: dict[str, str]
    feature_names: list[str]

    @classmethod
    def from_metadata(cls, evidence_metadata: dict[str, dict[str, Any]]) -> "ObservationSchema":
        root_ids = list(evidence_metadata.keys())
        slot_slices = {}
        data_types = {}
        possible_values = {}
        default_values = {}
        categorical_integer_roots = set()
        question_text = {}
        feature_names = [f"age_bin_{idx}" for idx in range(8)] + ["sex_M", "sex_F"]
        cursor = 10
        for root_id in root_ids:
            meta = evidence_metadata[root_id]
            data_type = meta.get("data_type", "B")
            raw_values = meta.get("possible-values", [])
            values = [str(value) for value in raw_values]
            default_value = meta.get("default_value")
            default_value = None if default_value is None else str(default_value)
            question_text[root_id] = meta.get("question_en", root_id)
            data_types[root_id] = data_type
            possible_values[root_id] = values
            default_values[root_id] = default_value
            if data_type == "B":
                slot_slices[root_id] = (cursor, cursor + 1)
                feature_names.append(root_id)
                cursor += 1
            elif data_type == "C":
                if raw_values and not isinstance(raw_values[0], str):
                    categorical_integer_roots.add(root_id)
                    slot_slices[root_id] = (cursor, cursor + 1)
                    feature_names.append(root_id)
                    cursor += 1
                else:
                    slot_slices[root_id] = (cursor, cursor + len(values))
                    feature_names.extend(f"{root_id}__{value}" for value in values)
                    cursor += len(values)
            elif data_type == "M":
                slot_slices[root_id] = (cursor, cursor + len(values))
                feature_names.extend(f"{root_id}__{value}" for value in values)
                cursor += len(values)
            else:
                raise ValueError(f"Unsupported evidence type {data_type} for {root_id}")
        return cls(root_ids, slot_slices, data_types, possible_values, default_values, categorical_integer_roots, question_text, feature_names)

    @property
    def feature_size(self) -> int:
        return len(self.feature_names)

    def initial_state(self, age: int, sex: str) -> np.ndarray:
        state = np.zeros(self.feature_size, dtype=np.float32)
        state[encode_age(int(age))] = 1.0
        state[8 + encode_sex(str(sex))] = 1.0
        return state

    def apply_root_observation(self, state: np.ndarray, root_id: str, present_values: list[str] | None = None) -> np.ndarray:
        if root_id not in self.slot_slices:
            return state
        values = [str(value) for value in (present_values or [])]
        data_type = self.data_types[root_id]
        start, end = self.slot_slices[root_id]
        default_value = self.default_values[root_id]
        if data_type == "B":
            state[start] = 1.0 if values else -1.0
            return state
        if root_id in self.categorical_integer_roots:
            chosen = values[0] if values else default_value
            if chosen is None:
                state[start] = -1.0
            else:
                possible = self.possible_values[root_id]
                denominator = max(1, len(possible) - 1)
                state[start] = float(possible.index(str(chosen))) / denominator if str(chosen) in possible else -1.0
            return state
        state[start:end] = -1.0
        if values:
            for value in values:
                if value in self.possible_values[root_id]:
                    state[start + self.possible_values[root_id].index(value)] = 1.0
        return state


def tokens_to_root_values(raw_tokens: Any) -> dict[str, list[str]]:
    root_values: dict[str, list[str]] = defaultdict(list)
    for token in safe_parse_list(raw_tokens):
        root_id, value = parse_evidence_token(str(token))
        root_values[root_id].append(PRESENT_STATE if value is None else str(value))
    return dict(root_values)


def roots_from_token_list(raw_tokens: Any) -> set[str]:
    return {parse_evidence_token(str(token))[0] for token in safe_parse_list(raw_tokens)}


def state_for_root_values(root_id: str, values: list[str], evidence_metadata: dict[str, dict[str, Any]]) -> str:
    if not values:
        return ABSENT_STATE
    data_type = evidence_metadata[root_id].get("data_type", "B")
    clean_values = [str(v) for v in values if str(v) != PRESENT_STATE]
    if data_type == "B":
        return PRESENT_STATE
    if data_type == "C":
        return clean_values[0] if clean_values else PRESENT_STATE
    if data_type == "M":
        return "|".join(sorted(set(clean_values))) if clean_values else PRESENT_STATE
    return clean_values[0] if clean_values else PRESENT_STATE


def values_for_state(root_id: str, state: str, evidence_metadata: dict[str, dict[str, Any]]) -> list[str]:
    if state == ABSENT_STATE:
        return []
    data_type = evidence_metadata[root_id].get("data_type", "B")
    if data_type == "B":
        return [PRESENT_STATE]
    if data_type == "M":
        return [v for v in state.split("|") if v]
    return [state]


def row_root_states(row: pd.Series | dict[str, Any], evidence_metadata: dict[str, dict[str, Any]]) -> dict[str, str]:
    getter = row.get if isinstance(row, dict) else row.__getitem__
    root_values = tokens_to_root_values(getter("EVIDENCES"))
    return {
        root_id: state_for_root_values(root_id, values, evidence_metadata)
        for root_id, values in root_values.items()
        if root_id in evidence_metadata
    }


def encode_observed_state(row: pd.Series | dict[str, Any], observed_states: dict[str, str], schema: ObservationSchema, evidence_metadata: dict[str, dict[str, Any]]) -> np.ndarray:
    getter = row.get if isinstance(row, dict) else row.__getitem__
    state = schema.initial_state(int(getter("AGE")), str(getter("SEX")))
    for root_id in sorted(observed_states):
        schema.apply_root_observation(state, root_id, values_for_state(root_id, observed_states[root_id], evidence_metadata))
    return state

## 2. Load Dataset, Reference Runs, And Partial-Evidence MLP

In [3]:
dataset_paths = ensure_dataset_present(DATASET_DIR)
evidence_metadata = load_json(dataset_paths["release_evidences.json"])
conditions_metadata = load_json(dataset_paths["release_conditions.json"])
schema = ObservationSchema.from_metadata(evidence_metadata)
root_ids = schema.root_ids
parent_root = {
    root_id: (meta.get("code_question") if meta.get("code_question") in evidence_metadata else root_id)
    for root_id, meta in evidence_metadata.items()
}

raw_train = attach_split_metadata(load_patient_split(dataset_paths[SPLIT_TO_FILENAME["train"]], nrows=TRAIN_LIKELIHOOD_NROWS), "train")
raw_validate_full = attach_split_metadata(load_patient_split(dataset_paths[SPLIT_TO_FILENAME["validate"]], nrows=CALIBRATION_ROWS), "validate")
raw_test = attach_split_metadata(load_patient_split(dataset_paths[SPLIT_TO_FILENAME["test"]]), "test")

notebook13_predictions = pd.read_csv(require_file(NOTEBOOK13_RUN / "predictions.csv", "Notebook 13 predictions"))
with (NOTEBOOK13_RUN / "metrics.json").open("r", encoding="utf-8") as handle:
    notebook13_metrics = json.load(handle)

selected_partial = load_json(require_file(PARTIAL_MLP_SELECTED, "selected partial-evidence model pointer"))
PARTIAL_MLP_DIR = Path(selected_partial["selected_artifact_dir"])
PARTIAL_MLP_CHECKPOINT = require_file(PARTIAL_MLP_DIR / "best_model.pt", "partial-evidence MLP checkpoint")

checkpoint = torch.load(PARTIAL_MLP_CHECKPOINT, map_location="cpu")
label_names = [str(x) for x in checkpoint["label_names"]]
pathology_to_index = {label: idx for idx, label in enumerate(label_names)}
if checkpoint["feature_names"] != schema.feature_names:
    raise ValueError("Current evidence schema feature names do not match the selected partial-evidence MLP checkpoint.")

# Primary 49-case scope exactly matches Notebook 13.
primary_case_ids = notebook13_predictions["case_id"].astype(str).tolist()
if SMOKE_MODE:
    primary_case_ids = primary_case_ids[:SMOKE_MAX_CASES]
primary_test_cases = raw_test[raw_test["case_id"].isin(primary_case_ids)].copy()
primary_test_cases["case_id"] = primary_test_cases["case_id"].astype(str)
primary_test_cases = primary_test_cases.set_index("case_id").loc[primary_case_ids].reset_index()

print("Train rows for likelihoods:", len(raw_train))
print("Validate rows for calibration:", len(raw_validate_full))
print("Primary test cases:", len(primary_test_cases))
print("Feature size:", schema.feature_size)
print("Labels:", len(label_names))
print("Notebook 13 reference:", {k: notebook13_metrics[k] for k in ["accuracy", "top5_accuracy", "mean_requests"]})

Train rows for likelihoods: 1025602
Validate rows for calibration: 10000
Primary test cases: 49
Feature size: 922
Labels: 49
Notebook 13 reference: {'accuracy': 0.8775510204081632, 'top5_accuracy': 0.9387755102040817, 'mean_requests': 6.591836734693878}


/var/folders/rg/t2plp1_x4lzdxjkl5fy9hcd80000gp/T/ipykernel_70914/4100177821.py:23: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(PARTIAL_MLP_CHECKPOI

## 3. Build Train-Only Bayesian Likelihood Tables

The policy can use only train-derived statistics. The test labels and differentials are not used here.

In [4]:
def normalized_entropy(probs: np.ndarray) -> float:
    probs = np.asarray(probs, dtype=np.float64)
    probs = probs / max(EPS, probs.sum())
    return float(-(probs * np.log(np.clip(probs, EPS, 1.0))).sum() / math.log(len(probs)))


def softmax_np(logits: np.ndarray) -> np.ndarray:
    logits = np.asarray(logits, dtype=np.float64)
    shifted = logits - np.max(logits)
    exp = np.exp(shifted)
    return exp / max(EPS, exp.sum())


def top_margin(probs: np.ndarray) -> float:
    ordered = np.sort(np.asarray(probs, dtype=np.float64))[::-1]
    return float(ordered[0] - ordered[1]) if len(ordered) >= 2 else 0.0


def top_k_indices(probs: np.ndarray, k: int = 5) -> list[int]:
    return np.argsort(np.asarray(probs))[::-1][:k].astype(int).tolist()


@dataclass
class BayesianStats:
    label_names: list[str]
    disease_prior: np.ndarray
    log_prior: np.ndarray
    age_likelihood: np.ndarray
    sex_likelihood: np.ndarray
    root_states: dict[str, list[str]]
    likelihood: dict[str, np.ndarray]
    log_likelihood: dict[str, np.ndarray]
    log_odds_support: dict[str, np.ndarray]
    global_state_probability: dict[str, np.ndarray]
    global_present_rate: dict[str, float]
    root_mi: dict[str, float]
    root_mi_norm: dict[str, float]
    train_rows: int


def build_bayesian_stats(train_df: pd.DataFrame) -> BayesianStats:
    start = time.time()
    disease_counts = Counter(str(x) for x in train_df["PATHOLOGY"])
    label_count = np.array([disease_counts[label] for label in label_names], dtype=np.float64)
    total = float(label_count.sum())
    disease_prior = (label_count + ALPHA) / (total + ALPHA * len(label_names))
    log_prior = np.log(np.clip(disease_prior, EPS, 1.0))

    age_counts = np.full((8, len(label_names)), ALPHA, dtype=np.float64)
    sex_counts = np.full((2, len(label_names)), ALPHA, dtype=np.float64)
    for row in train_df.itertuples(index=False):
        idx = pathology_to_index[str(row.PATHOLOGY)]
        age_counts[encode_age(int(row.AGE)), idx] += 1.0
        sex_counts[encode_sex(str(row.SEX)), idx] += 1.0
    age_likelihood = age_counts / age_counts.sum(axis=0, keepdims=True)
    sex_likelihood = sex_counts / sex_counts.sum(axis=0, keepdims=True)

    root_state_counts: dict[str, dict[str, Counter[str]]] = {
        root_id: {label: Counter({ABSENT_STATE: int(disease_counts[label])}) for label in label_names}
        for root_id in root_ids
    }
    states_seen: dict[str, set[str]] = {root_id: {ABSENT_STATE} for root_id in root_ids}

    for row in tqdm(train_df.itertuples(index=False), total=len(train_df), desc="Building root outcome counts"):
        label = str(row.PATHOLOGY)
        states = row_root_states(row._asdict(), evidence_metadata)
        for root_id, state in states.items():
            if root_id not in root_state_counts:
                continue
            counter = root_state_counts[root_id][label]
            counter[ABSENT_STATE] -= 1
            counter[state] += 1
            states_seen[root_id].add(state)

    root_states = {}
    likelihood = {}
    log_likelihood = {}
    log_odds_support = {}
    global_state_probability = {}
    global_present_rate = {}
    root_mi = {}

    for root_id in tqdm(root_ids, desc="Normalizing likelihood tables"):
        states = [ABSENT_STATE] + sorted(s for s in states_seen[root_id] if s != ABSENT_STATE)
        root_states[root_id] = states
        counts = np.zeros((len(states), len(label_names)), dtype=np.float64)
        for s_idx, state in enumerate(states):
            for d_idx, label in enumerate(label_names):
                counts[s_idx, d_idx] = root_state_counts[root_id][label].get(state, 0)
        denom = counts.sum(axis=0, keepdims=True) + ALPHA * len(states)
        probs = (counts + ALPHA) / np.clip(denom, EPS, None)
        likelihood[root_id] = probs
        log_likelihood[root_id] = np.clip(np.log(np.clip(probs, EPS, 1.0)), -LOG_LIKELIHOOD_CLIP, LOG_LIKELIHOOD_CLIP)
        global_probs = counts.sum(axis=1) / max(EPS, counts.sum())
        global_state_probability[root_id] = global_probs
        global_present_rate[root_id] = float(1.0 - global_probs[states.index(ABSENT_STATE)]) if ABSENT_STATE in states else 1.0

        # log-odds support of outcome for each disease versus all other diseases.
        odds = np.zeros_like(probs)
        for d_idx in range(len(label_names)):
            not_counts = counts.sum(axis=1) - counts[:, d_idx]
            not_total = max(EPS, float(not_counts.sum()))
            p_not = (not_counts + ALPHA) / (not_total + ALPHA * len(states))
            odds[:, d_idx] = np.log(np.clip(probs[:, d_idx], EPS, 1.0)) - np.log(np.clip(p_not, EPS, 1.0))
        log_odds_support[root_id] = np.clip(odds, -LOG_LIKELIHOOD_CLIP, LOG_LIKELIHOOD_CLIP)

        p_ds = counts / max(EPS, counts.sum())
        p_s = p_ds.sum(axis=1, keepdims=True)
        p_d = p_ds.sum(axis=0, keepdims=True)
        expected = p_s @ p_d
        mask = (p_ds > 0) & (expected > 0)
        root_mi[root_id] = float((p_ds[mask] * np.log(p_ds[mask] / expected[mask])).sum())

    max_mi = max(root_mi.values()) if root_mi else 1.0
    root_mi_norm = {root_id: float(root_mi[root_id] / max(max_mi, EPS)) for root_id in root_ids}

    stats = BayesianStats(
        label_names=label_names,
        disease_prior=disease_prior,
        log_prior=log_prior,
        age_likelihood=age_likelihood,
        sex_likelihood=sex_likelihood,
        root_states=root_states,
        likelihood=likelihood,
        log_likelihood=log_likelihood,
        log_odds_support=log_odds_support,
        global_state_probability=global_state_probability,
        global_present_rate=global_present_rate,
        root_mi=root_mi,
        root_mi_norm=root_mi_norm,
        train_rows=len(train_df),
    )
    print(f"Built Bayesian stats in {time.time() - start:.1f}s")
    return stats


bayes_stats = build_bayesian_stats(raw_train)

# Persist likelihood artifacts.
priors_df = pd.DataFrame({"pathology": label_names, "prior": bayes_stats.disease_prior})
priors_df.to_csv(ARTIFACT_ROOT / "diagnosis_priors.csv", index=False)

likelihood_rows = []
for root_id in root_ids:
    states = bayes_stats.root_states[root_id]
    for s_idx, state in enumerate(states):
        row = {
            "root_evidence_id": root_id,
            "question_en": schema.question_text[root_id],
            "outcome_state": state,
            "global_state_probability": float(bayes_stats.global_state_probability[root_id][s_idx]),
            "global_present_rate": bayes_stats.global_present_rate[root_id],
            "root_mi": bayes_stats.root_mi[root_id],
            "root_mi_norm": bayes_stats.root_mi_norm[root_id],
        }
        for d_idx, label in enumerate(label_names):
            row[f"p__{label}"] = float(bayes_stats.likelihood[root_id][s_idx, d_idx])
        likelihood_rows.append(row)
root_outcome_likelihoods = pd.DataFrame(likelihood_rows)
root_outcome_likelihoods.to_csv(ARTIFACT_ROOT / "root_outcome_likelihoods.csv", index=False)

root_information_stats = pd.DataFrame([
    {
        "root_evidence_id": root_id,
        "question_en": schema.question_text[root_id],
        "data_type": schema.data_types[root_id],
        "num_outcome_states": len(bayes_stats.root_states[root_id]),
        "global_present_rate": bayes_stats.global_present_rate[root_id],
        "root_mi": bayes_stats.root_mi[root_id],
        "root_mi_norm": bayes_stats.root_mi_norm[root_id],
    }
    for root_id in root_ids
]).sort_values("root_mi", ascending=False)
root_information_stats.to_csv(ARTIFACT_ROOT / "root_information_stats.csv", index=False)

# Normalization validation.
max_likelihood_sum_error = 0.0
for root_id in root_ids:
    sums = bayes_stats.likelihood[root_id].sum(axis=0)
    max_likelihood_sum_error = max(max_likelihood_sum_error, float(np.abs(sums - 1.0).max()))
print("Max likelihood column-sum error:", max_likelihood_sum_error)
display(root_information_stats.head(10))

Building root outcome counts:   0%|          | 0/1025602 [00:00<?, ?it/s]

Normalizing likelihood tables:   0%|          | 0/223 [00:00<?, ?it/s]

Built Bayesian stats in 42.9s
Max likelihood column-sum error: 9.749978602258125e-13


,root_evidence_id,question_en,data_type,num_outcome_states,global_present_rate,root_mi,root_mi_norm
1,E_55,Do you feel pain somewhere?,M,18196,0.778994,2.820390,1.000000
4,E_54,Characterize your pain:,M,666,0.778994,2.318533,0.822061
3,E_57,Does the pain radiate to another location?,M,1613,0.778994,1.376401,0.488018
5,E_59,How fast did the pain appear?,C,12,0.778994,0.870558,0.308666
9,E_133,Where is the affected region located?,M,9202,0.195078,0.852882,0.302399
6,E_56,How intense is the pain?,C,12,0.778994,0.818654,0.290263
7,E_58,How precisely is the pain located?,C,12,0.778994,0.802388,0.284495
11,E_130,What color is the rash?,C,6,0.195078,0.724455,0.256863
14,E_136,How severe is the itching?,C,12,0.195078,0.662398,0.234860
15,E_135,Is the lesion (or are the lesions) larger than 1cm?,C,3,0.195078,0.582233,0.206437


## 4. Partial-Evidence MLP And Posterior Fusion

In [5]:
class DirectDiagnosisMLP(nn.Module):
    def __init__(self, input_dim: int, hidden_sizes: list[int], num_classes: int, dropout: float = 0.0):
        super().__init__()
        layers = []
        previous_dim = input_dim
        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(previous_dim, hidden_size))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            previous_dim = hidden_size
        self.backbone = nn.Sequential(*layers)
        self.classifier = nn.Linear(previous_dim, num_classes)

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        return self.classifier(self.backbone(features))


def choose_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


partial_config = checkpoint["resolved_run_config"]
device = choose_device()
partial_mlp = DirectDiagnosisMLP(
    input_dim=schema.feature_size,
    hidden_sizes=[int(x) for x in partial_config["hidden_sizes"]],
    num_classes=len(label_names),
    dropout=float(partial_config.get("dropout", 0.0)),
).to(device)
partial_mlp.load_state_dict(checkpoint["model_state_dict"])
partial_mlp.eval()


def predict_mlp_proba(feature_matrix: np.ndarray, batch_size: int = 512) -> np.ndarray:
    if len(feature_matrix) == 0:
        return np.zeros((0, len(label_names)), dtype=np.float64)
    loader = DataLoader(TensorDataset(torch.tensor(feature_matrix, dtype=torch.float32)), batch_size=batch_size, shuffle=False)
    chunks = []
    with torch.no_grad():
        for (batch_x,) in loader:
            logits = partial_mlp(batch_x.to(device))
            chunks.append(torch.softmax(logits, dim=1).cpu().numpy())
    return np.vstack(chunks).astype(np.float64)


def fuse_posteriors(mlp_probs: np.ndarray, bayes_probs: np.ndarray) -> np.ndarray:
    log_mix = MLP_FUSION_WEIGHT * np.log(np.clip(mlp_probs, EPS, 1.0)) + BAYES_FUSION_WEIGHT * np.log(np.clip(bayes_probs, EPS, 1.0))
    return softmax_np(log_mix)


print("Partial MLP dir:", PARTIAL_MLP_DIR)
print("Partial MLP device:", device)
print("Partial MLP config test metrics:", partial_config.get("test_metrics"))

Partial MLP dir: /Users/bilalawan/claw/assignments/baseline_model/artifacts/one_shot_partial_evidence/partial_evidence_one_shot_final_policy_masked_v2
Partial MLP device: mps
Partial MLP config test metrics: {'accuracy': 0.515250762538127, 'top3_accuracy': 0.7413870693534677, 'top5_accuracy': 0.8273663683184159, 'macro_f1': 0.5187722515897814}


## 5. Bayesian Evidence Ledger And VOI Scoring

In [6]:
class BayesianEvidenceLedger:
    def __init__(self, row: pd.Series, bayes_stats: BayesianStats):
        self.row = row
        self.bayes_stats = bayes_stats
        self.case_id = str(row["case_id"])
        self.full_root_states = row_root_states(row, evidence_metadata)
        self.observed_states: dict[str, str] = {}
        self.initial_roots = roots_from_token_list(row["INITIAL_EVIDENCE"])
        for root_id in sorted(self.initial_roots):
            self.observed_states[root_id] = self.full_root_states.get(root_id, ABSENT_STATE)
        self.requested_roots: list[str] = []
        self.history: list[dict[str, Any]] = []

    @property
    def num_requests(self) -> int:
        return len(self.requested_roots)

    def visible_status(self, root_id: str) -> str:
        return "present" if self.observed_states.get(root_id, ABSENT_STATE) != ABSENT_STATE else "absent"

    def legal_roots(self) -> list[str]:
        legal = []
        observed = set(self.observed_states)
        for root_id in root_ids:
            if root_id in observed:
                continue
            parent = parent_root.get(root_id, root_id)
            if parent == root_id:
                legal.append(root_id)
                continue
            parent_present = self.observed_states.get(parent, ABSENT_STATE) != ABSENT_STATE
            implied_parent_present = any(parent_root.get(r, r) == parent and state != ABSENT_STATE for r, state in self.observed_states.items())
            if parent_present or implied_parent_present:
                legal.append(root_id)
        return legal

    def reveal_root(self, root_id: str) -> dict[str, Any]:
        state = self.full_root_states.get(root_id, ABSENT_STATE)
        self.observed_states[root_id] = state
        if root_id not in self.initial_roots:
            self.requested_roots.append(root_id)
        payload = {
            "root_evidence_id": root_id,
            "question_en": schema.question_text.get(root_id, root_id),
            "outcome_state": state,
            "status": "present" if state != ABSENT_STATE else "absent",
        }
        return payload

    def log_scores(self, extra_state: tuple[str, str] | None = None) -> np.ndarray:
        scores = self.bayes_stats.log_prior.copy()
        age_bin = encode_age(int(self.row["AGE"]))
        sex_bin = encode_sex(str(self.row["SEX"]))
        scores += AGE_SEX_WEIGHT * np.log(np.clip(self.bayes_stats.age_likelihood[age_bin], EPS, 1.0))
        scores += AGE_SEX_WEIGHT * np.log(np.clip(self.bayes_stats.sex_likelihood[sex_bin], EPS, 1.0))
        items = list(self.observed_states.items())
        if extra_state is not None:
            items.append(extra_state)
        for root_id, state in items:
            if root_id not in self.bayes_stats.root_states:
                continue
            states = self.bayes_stats.root_states[root_id]
            if state not in states:
                state = ABSENT_STATE if state == ABSENT_STATE else states[np.argmax(self.bayes_stats.global_state_probability[root_id])]
            s_idx = states.index(state)
            reliability = 0.25 + 0.75 * self.bayes_stats.root_mi_norm[root_id]
            scores += BAYES_EVIDENCE_WEIGHT * reliability * self.bayes_stats.log_likelihood[root_id][s_idx]
        return scores

    def bayes_posterior(self, extra_state: tuple[str, str] | None = None) -> np.ndarray:
        return softmax_np(self.log_scores(extra_state=extra_state))

    def mlp_feature(self, extra_state: tuple[str, str] | None = None) -> np.ndarray:
        observed = dict(self.observed_states)
        if extra_state is not None:
            observed[extra_state[0]] = extra_state[1]
        return encode_observed_state(self.row, observed, schema, evidence_metadata)

    def contradiction_score(self, diagnosis_idx: int, extra_state: tuple[str, str] | None = None) -> float:
        items = list(self.observed_states.items())
        if extra_state is not None:
            items.append(extra_state)
        score = 0.0
        for root_id, state in items:
            states = self.bayes_stats.root_states[root_id]
            if state not in states:
                continue
            s_idx = states.index(state)
            support = float(self.bayes_stats.log_odds_support[root_id][s_idx, diagnosis_idx])
            score += max(0.0, -support) * (0.25 + 0.75 * self.bayes_stats.root_mi_norm[root_id])
        return float(score)


def posterior_summary(probs: np.ndarray) -> dict[str, Any]:
    idxs = top_k_indices(probs, 5)
    return {
        "top1": label_names[idxs[0]],
        "top1_idx": int(idxs[0]),
        "confidence": float(probs[idxs[0]]),
        "margin": top_margin(probs),
        "entropy": normalized_entropy(probs),
        "top5": [label_names[i] for i in idxs],
        "top5_probs": [float(probs[i]) for i in idxs],
    }


def candidate_outcomes(root_id: str, belief: np.ndarray, max_outcomes: int = MAX_COUNTERFACTUAL_OUTCOMES) -> list[tuple[str, float]]:
    states = bayes_stats.root_states[root_id]
    p_outcome = bayes_stats.likelihood[root_id] @ belief
    p_outcome = p_outcome / max(EPS, p_outcome.sum())
    order = np.argsort(p_outcome)[::-1].tolist()
    chosen = []
    absent_idx = states.index(ABSENT_STATE) if ABSENT_STATE in states else None
    if absent_idx is not None:
        chosen.append(absent_idx)
    for idx in order:
        if idx not in chosen:
            chosen.append(idx)
        if len(chosen) >= max_outcomes:
            break
    mass = float(p_outcome[chosen].sum())
    return [(states[idx], float(p_outcome[idx] / max(EPS, mass))) for idx in chosen]


def rare_recovery_bonus(root_id: str, fused_probs: np.ndarray) -> float:
    active = top_k_indices(fused_probs, 5)
    best = 0.0
    for s_idx, state in enumerate(bayes_stats.root_states[root_id]):
        global_prob = float(bayes_stats.global_state_probability[root_id][s_idx])
        if state == ABSENT_STATE or global_prob > 0.05:
            continue
        for d_idx in active:
            support = float(bayes_stats.log_odds_support[root_id][s_idx, d_idx])
            if support >= 3.5:
                best = max(best, float(fused_probs[d_idx]) * min(1.0, (support - 3.5) / 4.0))
    return float(best)


def score_candidate_roots(ledger: BayesianEvidenceLedger, lambda_cost: float) -> tuple[pd.DataFrame, dict[str, Any]]:
    bayes_current = ledger.bayes_posterior()
    mlp_current = predict_mlp_proba(np.vstack([ledger.mlp_feature()]))[0]
    fused_current = fuse_posteriors(mlp_current, bayes_current)
    bayes_entropy = normalized_entropy(bayes_current)
    fused_entropy = normalized_entropy(fused_current)
    fused_margin = top_margin(fused_current)
    fused_top_idx = int(np.argmax(fused_current))
    current_contradiction = ledger.contradiction_score(fused_top_idx)

    legal = ledger.legal_roots()
    preliminary = []
    for root_id in legal:
        states = bayes_stats.root_states[root_id]
        p_outcome = bayes_stats.likelihood[root_id] @ bayes_current
        p_outcome = p_outcome / max(EPS, p_outcome.sum())
        expected_entropy = 0.0
        expected_margin = 0.0
        for s_idx, state in enumerate(states):
            p_state = float(p_outcome[s_idx])
            after = ledger.bayes_posterior(extra_state=(root_id, state))
            expected_entropy += p_state * normalized_entropy(after)
            expected_margin += p_state * top_margin(after)
        bayes_entropy_reduction = max(0.0, bayes_entropy - expected_entropy)
        preliminary.append({
            "root_evidence_id": root_id,
            "question_en": schema.question_text[root_id],
            "bayes_entropy_reduction": float(bayes_entropy_reduction),
            "bayes_expected_margin_gain": float(max(0.0, expected_margin - top_margin(bayes_current))),
            "root_mi_norm": bayes_stats.root_mi_norm[root_id],
            "global_present_rate": bayes_stats.global_present_rate[root_id],
            "num_outcome_states": len(states),
        })
    if not preliminary:
        context = {
            "bayes_probs": bayes_current,
            "mlp_probs": mlp_current,
            "fused_probs": fused_current,
            "max_utility": float("-inf"),
            "fused_summary": posterior_summary(fused_current),
            "bayes_summary": posterior_summary(bayes_current),
            "mlp_summary": posterior_summary(mlp_current),
            "contradiction_score": current_contradiction,
        }
        return pd.DataFrame(), context

    prelim_df = pd.DataFrame(preliminary).sort_values(
        ["bayes_entropy_reduction", "root_mi_norm", "root_evidence_id"], ascending=[False, False, True]
    )
    top_roots = prelim_df.head(TOP_BAYES_VOI_ROOTS)["root_evidence_id"].tolist()

    cf_rows = []
    cf_features = []
    for root_id in top_roots:
        # Use fused belief for counterfactual outcome probabilities because the final policy is fused.
        for state, p_state in candidate_outcomes(root_id, fused_current):
            bayes_after = ledger.bayes_posterior(extra_state=(root_id, state))
            cf_rows.append({"root_evidence_id": root_id, "outcome_state": state, "p_state": p_state, "bayes_after": bayes_after})
            cf_features.append(ledger.mlp_feature(extra_state=(root_id, state)))
    mlp_after_matrix = predict_mlp_proba(np.vstack(cf_features)) if cf_features else np.zeros((0, len(label_names)))

    by_root: dict[str, list[dict[str, Any]]] = defaultdict(list)
    for row, mlp_after in zip(cf_rows, mlp_after_matrix):
        fused_after = fuse_posteriors(mlp_after, row["bayes_after"])
        root_id = row["root_evidence_id"]
        by_root[root_id].append({
            "p_state": float(row["p_state"]),
            "outcome_state": row["outcome_state"],
            "mlp_after": mlp_after,
            "bayes_after": row["bayes_after"],
            "fused_after": fused_after,
            "fused_entropy_after": normalized_entropy(fused_after),
            "fused_margin_after": top_margin(fused_after),
            "contradiction_after": ledger.contradiction_score(fused_top_idx, extra_state=(root_id, row["outcome_state"])),
        })

    detailed_rows = []
    prelim_lookup = {row["root_evidence_id"]: row for row in prelim_df.to_dict(orient="records")}
    for root_id in top_roots:
        outcomes = by_root[root_id]
        p_total = max(EPS, sum(x["p_state"] for x in outcomes))
        expected_fused_entropy = sum((x["p_state"] / p_total) * x["fused_entropy_after"] for x in outcomes)
        expected_margin = sum((x["p_state"] / p_total) * x["fused_margin_after"] for x in outcomes)
        expected_contradiction = sum((x["p_state"] / p_total) * x["contradiction_after"] for x in outcomes)
        entropy_reduction = max(0.0, fused_entropy - expected_fused_entropy)
        margin_gain = max(0.0, expected_margin - fused_margin)
        contradiction_gain = max(0.0, current_contradiction - expected_contradiction) / max(1.0, current_contradiction)
        rare_bonus = rare_recovery_bonus(root_id, fused_current)
        redundancy_penalty = 0.02 * float(bayes_stats.global_present_rate[root_id] > 0.55) + 0.02 * float(bayes_stats.root_mi_norm[root_id] < 0.05)
        utility = (
            UTILITY_WEIGHTS["expected_fused_entropy_reduction"] * entropy_reduction
            + UTILITY_WEIGHTS["expected_margin_gain"] * margin_gain
            + UTILITY_WEIGHTS["contradiction_resolution_gain"] * contradiction_gain
            + UTILITY_WEIGHTS["rare_recovery_bonus"] * rare_bonus
            - float(lambda_cost)
            - redundancy_penalty
        )
        detailed_rows.append({
            **prelim_lookup[root_id],
            "expected_fused_entropy_reduction": float(entropy_reduction),
            "expected_margin_gain": float(margin_gain),
            "contradiction_resolution_gain": float(contradiction_gain),
            "rare_recovery_bonus": float(rare_bonus),
            "redundancy_penalty": float(redundancy_penalty),
            "lambda_cost": float(lambda_cost),
            "utility": float(utility),
            "expected_fused_entropy_after": float(expected_fused_entropy),
            "expected_fused_margin_after": float(expected_margin),
            "expected_contradiction_after": float(expected_contradiction),
            "outcome_states_considered": len(outcomes),
        })

    scores = pd.DataFrame(detailed_rows).sort_values(
        ["utility", "expected_fused_entropy_reduction", "root_evidence_id"], ascending=[False, False, True]
    ).reset_index(drop=True)
    scores["voi_rank"] = np.arange(1, len(scores) + 1)
    context = {
        "bayes_probs": bayes_current,
        "mlp_probs": mlp_current,
        "fused_probs": fused_current,
        "max_utility": float(scores["utility"].max()) if len(scores) else float("-inf"),
        "fused_summary": posterior_summary(fused_current),
        "bayes_summary": posterior_summary(bayes_current),
        "mlp_summary": posterior_summary(mlp_current),
        "contradiction_score": float(current_contradiction),
    }
    return scores, context


def stop_certificate(context: dict[str, Any], request_count: int) -> tuple[bool, str]:
    fused = context["fused_summary"]
    bayes = context["bayes_summary"]
    mlp = context["mlp_summary"]
    agreement_ok = (
        fused["top1"] == bayes["top1"]
        or fused["top1"] == mlp["top1"]
        or bayes["top1"] in mlp["top5"]
        or mlp["top1"] in bayes["top5"]
    )
    checks = {
        "min_requests": request_count >= STOP_CONFIG["min_requests"],
        "confidence": fused["confidence"] >= STOP_CONFIG["fused_confidence_min"],
        "margin": fused["margin"] >= STOP_CONFIG["fused_margin_min"],
        "entropy": fused["entropy"] <= STOP_CONFIG["fused_entropy_max"],
        "remaining_utility": context["max_utility"] <= 0.0,
        "contradiction": context["contradiction_score"] <= STOP_CONFIG["contradiction_max"],
        "agreement": agreement_ok,
    }
    if all(checks.values()):
        return True, "bayesian_voi_stop_certificate"
    failed = [name for name, ok in checks.items() if not ok]
    return False, "continue_failed_" + "+".join(failed)

## 6. Posterior Calibration And Notebook 13 Replay Diagnostics

In [7]:
def evaluate_probs(probs: np.ndarray, true_indices: np.ndarray) -> dict[str, float]:
    pred = probs.argmax(axis=1)
    top3 = np.argsort(probs, axis=1)[:, ::-1][:, :3]
    top5 = np.argsort(probs, axis=1)[:, ::-1][:, :5]
    return {
        "accuracy": float(np.mean(pred == true_indices)),
        "top3_accuracy": float(np.mean([y in row for y, row in zip(true_indices, top3)])),
        "top5_accuracy": float(np.mean([y in row for y, row in zip(true_indices, top5)])),
        "macro_f1": float(f1_score(true_indices, pred, average="macro")),
    }


calibration_rows = []
for row in tqdm(raw_validate_full.itertuples(index=False), total=len(raw_validate_full), desc="Calibrating initial Bayesian posterior"):
    row_s = pd.Series(row._asdict())
    if str(row_s["PATHOLOGY"]) not in pathology_to_index:
        continue
    ledger = BayesianEvidenceLedger(row_s, bayes_stats)
    bayes_probs = ledger.bayes_posterior()
    feature = ledger.mlp_feature()
    mlp_probs = predict_mlp_proba(np.vstack([feature]))[0]
    fused_probs = fuse_posteriors(mlp_probs, bayes_probs)
    y = pathology_to_index[str(row_s["PATHOLOGY"])]
    calibration_rows.append({
        "case_id": row_s["case_id"],
        "true_pathology": row_s["PATHOLOGY"],
        "bayes_top1": label_names[int(np.argmax(bayes_probs))],
        "mlp_top1": label_names[int(np.argmax(mlp_probs))],
        "fused_top1": label_names[int(np.argmax(fused_probs))],
        "bayes_correct": int(np.argmax(bayes_probs) == y),
        "mlp_correct": int(np.argmax(mlp_probs) == y),
        "fused_correct": int(np.argmax(fused_probs) == y),
        "bayes_confidence": float(np.max(bayes_probs)),
        "mlp_confidence": float(np.max(mlp_probs)),
        "fused_confidence": float(np.max(fused_probs)),
        "bayes_entropy": normalized_entropy(bayes_probs),
        "mlp_entropy": normalized_entropy(mlp_probs),
        "fused_entropy": normalized_entropy(fused_probs),
    })
posterior_calibration = pd.DataFrame(calibration_rows)
posterior_calibration.to_csv(ARTIFACT_ROOT / "posterior_calibration.csv", index=False)
print("Calibration rows:", len(posterior_calibration))
if len(posterior_calibration):
    display(posterior_calibration[["bayes_correct", "mlp_correct", "fused_correct"]].mean().to_frame("initial_accuracy"))


def load_trace_records(path: Path) -> list[dict[str, Any]]:
    records = []
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            if line.strip():
                records.append(json.loads(line))
    return records


trace_records = load_trace_records(require_file(NOTEBOOK13_RUN / "traces.jsonl", "Notebook 13 traces"))
trace_records = [r for r in trace_records if str(r.get("case_id")) in set(primary_case_ids)]
test_lookup = {str(row.case_id): pd.Series(row._asdict()) for row in raw_test.itertuples(index=False)}

replay_rows = []
for record in tqdm(trace_records, desc="Replaying Notebook 13 through Bayesian ledger"):
    case_id = str(record["case_id"])
    ledger = BayesianEvidenceLedger(test_lookup[case_id], bayes_stats)
    for turn in record.get("trace", []):
        scores, context = score_candidate_roots(ledger, lambda_cost=0.05)
        top = scores.iloc[0].to_dict() if len(scores) else {}
        requested_root = None
        reveal = turn.get("reveal_payload") or {}
        if reveal.get("root_evidence_id"):
            requested_root = str(reveal["root_evidence_id"])
        request_rank = None
        request_utility = None
        if requested_root and len(scores):
            match = scores[scores["root_evidence_id"] == requested_root]
            if len(match):
                request_rank = int(match.iloc[0]["voi_rank"])
                request_utility = float(match.iloc[0]["utility"])
        replay_rows.append({
            "case_id": case_id,
            "turn_index": int(turn.get("turn_index", len(replay_rows) + 1)),
            "true_pathology": record.get("true_pathology"),
            "notebook13_prediction": record.get("predicted_pathology"),
            "notebook13_correct": bool(record.get("predicted_pathology") == record.get("true_pathology")),
            "requests_before": ledger.num_requests,
            "fused_top1": context["fused_summary"]["top1"],
            "bayes_top1": context["bayes_summary"]["top1"],
            "mlp_top1": context["mlp_summary"]["top1"],
            "fused_confidence": context["fused_summary"]["confidence"],
            "fused_entropy": context["fused_summary"]["entropy"],
            "fused_margin": context["fused_summary"]["margin"],
            "contradiction_score": context["contradiction_score"],
            "max_remaining_utility": context["max_utility"],
            "top_voi_root": top.get("root_evidence_id"),
            "top_voi_question": top.get("question_en"),
            "top_voi_utility": top.get("utility"),
            "actual_requested_root": requested_root,
            "actual_request_voi_rank": request_rank,
            "actual_request_utility": request_utility,
        })
        if requested_root:
            ledger.reveal_root(requested_root)

notebook13_replay_bayesian_diagnostics = pd.DataFrame(replay_rows)
notebook13_replay_bayesian_diagnostics.to_csv(ARTIFACT_ROOT / "notebook13_replay_bayesian_diagnostics.csv", index=False)
display(notebook13_replay_bayesian_diagnostics.head())

Calibrating initial Bayesian posterior:   0%|          | 0/10000 [00:00<?, ?it/s]

Calibration rows: 10000


,initial_accuracy
bayes_correct,0.2368
mlp_correct,0.3693
fused_correct,0.3682


Replaying Notebook 13 through Bayesian ledger:   0%|          | 0/49 [00:00<?, ?it/s]

/var/folders/rg/t2plp1_x4lzdxjkl5fy9hcd80000gp/T/ipykernel_70914/1382728065.py:153: RuntimeWarning: divide by zero encountered in matmul
  p_outcome = bayes_stats.likelihood[root_id] @ bayes_current
/var/folders/rg/t2plp1_x4lzdxjkl5fy9hcd80000gp/T/ipykernel_70914/1382728065.py:153: RuntimeWarning: overflow encountered in matmul
  p_outcome = bayes_stats.likelihood[root_id] @ bayes_current
/var/folders/rg/t2plp1_x4lzdxjkl5fy9hcd80000gp/T/ipykernel_70914/1382728065.py:153: RuntimeWarning: invalid value encountered in matmul
  p_outcome = bayes_stats.likelihood[root_id] @ bayes_current
/var/folders/rg/t2plp1_x4lzdxjkl5fy9hcd80000gp/T/ipykernel_70914/1382728065.py:153: RuntimeWarning: divide by zero encountered in matmul
  p_outcome = bayes_stats.likelihood[root_id] @ bayes_current
/var/folders/rg/t2plp1_x4lzdxjkl5fy9hcd80000gp/T/ipykernel_70914/1382728065.py:153: RuntimeWarning: overflow encountered in matmul
  p_outcome = bayes_stats.likelihood[root_id] @ bayes_current
/var/folders/rg/t2

,case_id,turn_index,true_pathology,notebook13_prediction,notebook13_correct,requests_before,fused_top1,bayes_top1,mlp_top1,fused_confidence,fused_entropy,fused_margin,contradiction_score,max_remaining_utility,top_voi_root,top_voi_question,top_voi_utility,actual_requested_root,actual_request_voi_rank,actual_request_utility
0,test:38475,1,Acute COPD exacerbation / infection,Myocarditis,False,0,Anemia,Anemia,Bronchospasm / acute asthma exacerbation,0.073064,0.860613,0.009733,0.0,0.006921,E_201,Do you have a cough?,0.006921,E_91,21.0,-0.034371
1,test:38475,2,Acute COPD exacerbation / infection,Myocarditis,False,1,Anemia,Anemia,Bronchospasm / acute asthma exacerbation,0.076112,0.847699,0.012114,0.0,0.023604,E_155,"Do you feel your heart is beating fast (racing), irregularly (missing a beat) or do you feel palpitations?",0.023604,E_124,14.0,-0.017822
2,test:38475,3,Acute COPD exacerbation / infection,Myocarditis,False,2,Anemia,Anemia,Acute COPD exacerbation / infection,0.088255,0.845823,0.011599,0.0,0.013249,E_220,Do you have pain that is increased when you breathe in deeply?,0.013249,E_79,9.0,-0.007381
3,test:38475,4,Acute COPD exacerbation / infection,Myocarditis,False,3,Anemia,Anemia,Acute dystonic reactions,0.105623,0.817308,0.023082,0.0,0.077489,E_155,"Do you feel your heart is beating fast (racing), irregularly (missing a beat) or do you feel palpitations?",0.077489,E_151,11.0,-0.006113
4,test:38475,5,Acute COPD exacerbation / infection,Myocarditis,False,4,Anemia,Anemia,Anemia,0.138984,0.784053,0.052681,0.0,0.035621,E_155,"Do you feel your heart is beating fast (racing), irregularly (missing a beat) or do you feel palpitations?",0.035621,E_0,NaN,NaN


## 7. Offline Bayesian VOI Agent Sweep

This is the main experiment. The agent starts from initial evidence, scores legal hidden evidence fields, requests the best field, and stops using a VOI stop certificate.

In [8]:
def run_bayesian_voi_agent(lambda_cost: float, case_frame: pd.DataFrame) -> tuple[pd.DataFrame, list[dict[str, Any]], pd.DataFrame]:
    prediction_rows = []
    trace_records_out = []
    candidate_rows = []

    for row in tqdm(case_frame.itertuples(index=False), total=len(case_frame), desc=f"lambda={lambda_cost:.2f}"):
        row_s = pd.Series(row._asdict())
        case_id = str(row_s["case_id"])
        ledger = BayesianEvidenceLedger(row_s, bayes_stats)
        trace = []
        stop_reason = "max_requests_reached"
        final_context = None
        final_scores = None

        for turn_index in range(1, MAX_REQUEST_CAP + 1):
            scores, context = score_candidate_roots(ledger, lambda_cost=lambda_cost)
            final_context = context
            final_scores = scores
            should_stop, stop_detail = stop_certificate(context, ledger.num_requests)
            top_candidates = scores.head(20).copy() if len(scores) else pd.DataFrame()
            for _, cand in top_candidates.iterrows():
                candidate_rows.append({
                    "lambda_cost": float(lambda_cost),
                    "case_id": case_id,
                    "turn_index": turn_index,
                    "requests_before": ledger.num_requests,
                    **cand.to_dict(),
                })

            turn_record = {
                "turn_index": turn_index,
                "requests_before": ledger.num_requests,
                "bayes_summary": context["bayes_summary"],
                "mlp_summary": context["mlp_summary"],
                "fused_summary": context["fused_summary"],
                "contradiction_score": context["contradiction_score"],
                "max_remaining_utility": context["max_utility"],
                "stop_detail": stop_detail,
                "top_candidates": top_candidates[[
                    "voi_rank", "root_evidence_id", "question_en", "utility", "expected_fused_entropy_reduction",
                    "expected_margin_gain", "contradiction_resolution_gain", "rare_recovery_bonus", "lambda_cost"
                ]].to_dict(orient="records") if len(top_candidates) else [],
            }

            if should_stop:
                stop_reason = stop_detail
                turn_record["decision"] = "stop"
                trace.append(turn_record)
                break
            if len(scores) == 0:
                stop_reason = "no_legal_candidates"
                turn_record["decision"] = "stop_no_legal_candidates"
                trace.append(turn_record)
                break

            chosen = scores.iloc[0].to_dict()
            reveal = ledger.reveal_root(str(chosen["root_evidence_id"]))
            turn_record["decision"] = "request"
            turn_record["requested_root"] = chosen["root_evidence_id"]
            turn_record["requested_question"] = chosen["question_en"]
            turn_record["requested_utility"] = float(chosen["utility"])
            turn_record["reveal_payload"] = reveal
            trace.append(turn_record)
        else:
            # Recompute context at cap after final reveal.
            _, final_context = score_candidate_roots(ledger, lambda_cost=lambda_cost)

        if final_context is None:
            _, final_context = score_candidate_roots(ledger, lambda_cost=lambda_cost)
        bayes_probs = final_context["bayes_probs"]
        mlp_probs = final_context["mlp_probs"]
        fused_probs = final_context["fused_probs"]
        true_idx = pathology_to_index[str(row_s["PATHOLOGY"])]
        fused_top5 = [label_names[i] for i in top_k_indices(fused_probs, 5)]
        bayes_top5 = [label_names[i] for i in top_k_indices(bayes_probs, 5)]
        mlp_top5 = [label_names[i] for i in top_k_indices(mlp_probs, 5)]
        pred_row = {
            "lambda_cost": float(lambda_cost),
            "case_id": case_id,
            "source_row_index": int(row_s["source_row_index"]),
            "AGE": int(row_s["AGE"]),
            "SEX": str(row_s["SEX"]),
            "true_pathology": str(row_s["PATHOLOGY"]),
            "fused_predicted_pathology": label_names[int(np.argmax(fused_probs))],
            "bayes_predicted_pathology": label_names[int(np.argmax(bayes_probs))],
            "mlp_predicted_pathology": label_names[int(np.argmax(mlp_probs))],
            "fused_correct": bool(int(np.argmax(fused_probs)) == true_idx),
            "bayes_correct": bool(int(np.argmax(bayes_probs)) == true_idx),
            "mlp_correct": bool(int(np.argmax(mlp_probs)) == true_idx),
            "fused_top3_correct": bool(true_idx in top_k_indices(fused_probs, 3)),
            "fused_top5_correct": bool(true_idx in top_k_indices(fused_probs, 5)),
            "bayes_top5_correct": bool(true_idx in top_k_indices(bayes_probs, 5)),
            "mlp_top5_correct": bool(true_idx in top_k_indices(mlp_probs, 5)),
            "fused_ranked_differential": json.dumps(fused_top5, ensure_ascii=False),
            "bayes_ranked_differential": json.dumps(bayes_top5, ensure_ascii=False),
            "mlp_ranked_differential": json.dumps(mlp_top5, ensure_ascii=False),
            "num_requests": int(ledger.num_requests),
            "visible_root_count": int(len(ledger.observed_states)),
            "stop_reason": stop_reason,
            "final_fused_confidence": float(np.max(fused_probs)),
            "final_fused_margin": top_margin(fused_probs),
            "final_fused_entropy": normalized_entropy(fused_probs),
            "final_bayes_confidence": float(np.max(bayes_probs)),
            "final_bayes_entropy": normalized_entropy(bayes_probs),
            "final_mlp_confidence": float(np.max(mlp_probs)),
            "final_mlp_entropy": normalized_entropy(mlp_probs),
            "final_contradiction_score": float(final_context["contradiction_score"]),
            "final_max_remaining_utility": float(final_context["max_utility"]),
            "requested_roots": json.dumps(ledger.requested_roots, ensure_ascii=False),
        }
        prediction_rows.append(pred_row)
        trace_records_out.append({
            "lambda_cost": float(lambda_cost),
            "case_id": case_id,
            "true_pathology": str(row_s["PATHOLOGY"]),
            "fused_predicted_pathology": pred_row["fused_predicted_pathology"],
            "fused_correct": pred_row["fused_correct"],
            "num_requests": pred_row["num_requests"],
            "stop_reason": stop_reason,
            "trace": trace,
        })

    return pd.DataFrame(prediction_rows), trace_records_out, pd.DataFrame(candidate_rows)


all_predictions = []
all_traces = []
all_candidate_scores = []
start = time.time()
for lambda_cost in LAMBDA_COSTS:
    preds, traces, candidates = run_bayesian_voi_agent(lambda_cost, primary_test_cases)
    all_predictions.append(preds)
    all_traces.extend(traces)
    all_candidate_scores.append(candidates)
print(f"Offline agent sweep completed in {time.time() - start:.1f}s")

offline_agent_predictions = pd.concat(all_predictions, ignore_index=True)
voi_candidate_scores_top20 = pd.concat(all_candidate_scores, ignore_index=True) if all_candidate_scores else pd.DataFrame()
offline_agent_predictions.to_csv(ARTIFACT_ROOT / "offline_agent_predictions.csv", index=False)
voi_candidate_scores_top20.to_csv(ARTIFACT_ROOT / "voi_candidate_scores_top20.csv", index=False)
with (ARTIFACT_ROOT / "offline_agent_traces.jsonl").open("w", encoding="utf-8") as handle:
    for record in all_traces:
        handle.write(json.dumps(record, ensure_ascii=False) + "\n")

display(offline_agent_predictions.head())

lambda=0.00:   0%|          | 0/49 [00:00<?, ?it/s]

/var/folders/rg/t2plp1_x4lzdxjkl5fy9hcd80000gp/T/ipykernel_70914/1382728065.py:153: RuntimeWarning: divide by zero encountered in matmul
  p_outcome = bayes_stats.likelihood[root_id] @ bayes_current
/var/folders/rg/t2plp1_x4lzdxjkl5fy9hcd80000gp/T/ipykernel_70914/1382728065.py:153: RuntimeWarning: overflow encountered in matmul
  p_outcome = bayes_stats.likelihood[root_id] @ bayes_current
/var/folders/rg/t2plp1_x4lzdxjkl5fy9hcd80000gp/T/ipykernel_70914/1382728065.py:153: RuntimeWarning: invalid value encountered in matmul
  p_outcome = bayes_stats.likelihood[root_id] @ bayes_current
/var/folders/rg/t2plp1_x4lzdxjkl5fy9hcd80000gp/T/ipykernel_70914/1382728065.py:153: RuntimeWarning: divide by zero encountered in matmul
  p_outcome = bayes_stats.likelihood[root_id] @ bayes_current
/var/folders/rg/t2plp1_x4lzdxjkl5fy9hcd80000gp/T/ipykernel_70914/1382728065.py:153: RuntimeWarning: overflow encountered in matmul
  p_outcome = bayes_stats.likelihood[root_id] @ bayes_current
/var/folders/rg/t2

lambda=0.02:   0%|          | 0/49 [00:00<?, ?it/s]

/var/folders/rg/t2plp1_x4lzdxjkl5fy9hcd80000gp/T/ipykernel_70914/1382728065.py:153: RuntimeWarning: divide by zero encountered in matmul
  p_outcome = bayes_stats.likelihood[root_id] @ bayes_current
/var/folders/rg/t2plp1_x4lzdxjkl5fy9hcd80000gp/T/ipykernel_70914/1382728065.py:153: RuntimeWarning: overflow encountered in matmul
  p_outcome = bayes_stats.likelihood[root_id] @ bayes_current
/var/folders/rg/t2plp1_x4lzdxjkl5fy9hcd80000gp/T/ipykernel_70914/1382728065.py:153: RuntimeWarning: invalid value encountered in matmul
  p_outcome = bayes_stats.likelihood[root_id] @ bayes_current
/var/folders/rg/t2plp1_x4lzdxjkl5fy9hcd80000gp/T/ipykernel_70914/1382728065.py:153: RuntimeWarning: divide by zero encountered in matmul
  p_outcome = bayes_stats.likelihood[root_id] @ bayes_current
/var/folders/rg/t2plp1_x4lzdxjkl5fy9hcd80000gp/T/ipykernel_70914/1382728065.py:153: RuntimeWarning: overflow encountered in matmul
  p_outcome = bayes_stats.likelihood[root_id] @ bayes_current
/var/folders/rg/t2

lambda=0.05:   0%|          | 0/49 [00:00<?, ?it/s]

...

/var/folders/rg/t2plp1_x4lzdxjkl5fy9hcd80000gp/T/ipykernel_70914/1382728065.py:153: RuntimeWarning: divide by zero encountered in matmul
  p_outcome = bayes_stats.likelihood[root_id] @ bayes_current
/var/folders/rg/t2plp1_x4lzdxjkl5fy9hcd80000gp/T/ipykernel_70914/1382728065.py:153: RuntimeWarning: overflow encountered in matmul
  p_outcome = bayes_stats.likelihood[root_id] @ bayes_current
/var/folders/rg/t2plp1_x4lzdxjkl5fy9hcd80000gp/T/ipykernel_70914/1382728065.py:153: RuntimeWarning: invalid value encountered in matmul
  p_outcome = bayes_stats.likelihood[root_id] @ bayes_current
/var/folders/rg/t2plp1_x4lzdxjkl5fy9hcd80000gp/T/ipykernel_70914/1382728065.py:153: RuntimeWarning: divide by zero encountered in matmul
  p_outcome = bayes_stats.likelihood[root_id] @ bayes_current
/var/folders/rg/t2plp1_x4lzdxjkl5fy9hcd80000gp/T/ipykernel_70914/1382728065.py:153: RuntimeWarning: overflow encountered in matmul
  p_outcome = bayes_stats.likelihood[root_id] @ bayes_current
/var/folders/rg/t2

lambda=0.15:   0%|          | 0/49 [00:00<?, ?it/s]

/var/folders/rg/t2plp1_x4lzdxjkl5fy9hcd80000gp/T/ipykernel_70914/1382728065.py:153: RuntimeWarning: divide by zero encountered in matmul
  p_outcome = bayes_stats.likelihood[root_id] @ bayes_current
/var/folders/rg/t2plp1_x4lzdxjkl5fy9hcd80000gp/T/ipykernel_70914/1382728065.py:153: RuntimeWarning: overflow encountered in matmul
  p_outcome = bayes_stats.likelihood[root_id] @ bayes_current
/var/folders/rg/t2plp1_x4lzdxjkl5fy9hcd80000gp/T/ipykernel_70914/1382728065.py:153: RuntimeWarning: invalid value encountered in matmul
  p_outcome = bayes_stats.likelihood[root_id] @ bayes_current
/var/folders/rg/t2plp1_x4lzdxjkl5fy9hcd80000gp/T/ipykernel_70914/1382728065.py:153: RuntimeWarning: divide by zero encountered in matmul
  p_outcome = bayes_stats.likelihood[root_id] @ bayes_current
/var/folders/rg/t2plp1_x4lzdxjkl5fy9hcd80000gp/T/ipykernel_70914/1382728065.py:153: RuntimeWarning: overflow encountered in matmul
  p_outcome = bayes_stats.likelihood[root_id] @ bayes_current
/var/folders/rg/t2

Offline agent sweep completed in 992.5s


,lambda_cost,case_id,source_row_index,AGE,SEX,true_pathology,fused_predicted_pathology,bayes_predicted_pathology,mlp_predicted_pathology,fused_correct,...,final_fused_confidence,final_fused_margin,final_fused_entropy,final_bayes_confidence,final_bayes_entropy,final_mlp_confidence,final_mlp_entropy,final_contradiction_score,final_max_remaining_utility,requested_roots
0,0.0,test:38475,38475,56,M,Acute COPD exacerbation / infection,Acute dystonic reactions,Bronchiectasis,Acute dystonic reactions,False,...,0.988313,0.983118,0.020619,0.097343,0.874886,0.999655,8.829236e-04,1.214551,0.000000,"[""E_201"", ""E_155"", ""E_220"", ""E_125"", ""E_226"", ""E_89"", ""E_51"", ""E_91"", ""E_194"", ""E_181"", ""E_82"", ""E_162"", ""E_79"", ""E_222"", ""E_116"", ""E_104"", ""E_123"", ""E_69"",..."
1,0.0,test:104857,104857,18,M,Acute dystonic reactions,Acute dystonic reactions,Acute dystonic reactions,Acute dystonic reactions,True,...,0.999323,0.998873,0.001613,0.140378,0.854957,0.999994,1.987916e-05,0.000000,0.000000,"[""E_155"", ""E_218"", ""E_201"", ""E_194"", ""E_226"", ""E_220"", ""E_91"", ""E_181"", ""E_89"", ""E_116"", ""E_82"", ""E_79"", ""E_51"", ""E_123"", ""E_118"", ""E_69"", ""E_104"", ""E_49"", ..."
2,0.0,test:58986,58986,0,F,Acute laryngitis,Acute laryngitis,Acute laryngitis,Acute laryngitis,True,...,0.999958,0.999942,0.000140,0.389017,0.630970,1.000000,5.095615e-07,0.000000,0.000000,"[""E_194"", ""E_181"", ""E_155"", ""E_66"", ""E_125"", ""E_148"", ""E_124"", ""E_77"", ""E_144"", ""E_78"", ""E_151"", ""E_105"", ""E_129"", ""E_49"", ""E_208"", ""E_82"", ""E_220"", ""E_51"",..."
3,0.0,test:33118,33118,77,F,Acute otitis media,Viral pharyngitis,Viral pharyngitis,Viral pharyngitis,False,...,0.998905,0.998636,0.002826,0.213232,0.781204,0.999989,4.006812e-05,0.371962,0.000034,"[""E_50"", ""E_124"", ""E_148"", ""E_120"", ""E_78"", ""E_105"", ""E_97"", ""E_226"", ""E_155"", ""E_77"", ""E_49"", ""E_66"", ""E_144"", ""E_116"", ""E_125"", ""E_218"", ""E_214"", ""E_88"", ..."
4,0.0,test:93930,93930,42,M,Acute pulmonary edema,Acute pulmonary edema,Acute pulmonary edema,Acute pulmonary edema,True,...,0.999606,0.999419,0.001030,0.894142,0.162267,0.999973,8.814153e-05,0.000000,0.000000,"[""E_155"", ""E_148"", ""E_201"", ""E_77"", ""E_66"", ""E_144"", ""E_129"", ""E_70"", ""E_78"", ""E_125"", ""E_41"", ""E_91"", ""E_105"", ""E_218"", ""E_76"", ""E_194"", ""E_181"", ""E_89"", ""..."


## 8. Metrics, Comparisons, Figures, And Promotion Decision

In [9]:
def summarize_policy_group(group: pd.DataFrame) -> dict[str, Any]:
    true_indices = group["true_pathology"].map(pathology_to_index).to_numpy(dtype=int)
    # Reconstruct top-k from stored booleans for fused; direct probabilities are not stored to keep CSV compact.
    pred_indices = group["fused_predicted_pathology"].map(pathology_to_index).to_numpy(dtype=int)
    bayes_pred_indices = group["bayes_predicted_pathology"].map(pathology_to_index).to_numpy(dtype=int)
    mlp_pred_indices = group["mlp_predicted_pathology"].map(pathology_to_index).to_numpy(dtype=int)
    return {
        "lambda_cost": float(group["lambda_cost"].iloc[0]),
        "num_cases": int(len(group)),
        "fused_correct_count": int(group["fused_correct"].sum()),
        "fused_accuracy": float(group["fused_correct"].mean()),
        "fused_top3_accuracy": float(group["fused_top3_correct"].mean()),
        "fused_top5_accuracy": float(group["fused_top5_correct"].mean()),
        "fused_macro_f1": float(f1_score(true_indices, pred_indices, average="macro")),
        "bayes_accuracy": float(group["bayes_correct"].mean()),
        "bayes_top5_accuracy": float(group["bayes_top5_correct"].mean()),
        "bayes_macro_f1": float(f1_score(true_indices, bayes_pred_indices, average="macro")),
        "mlp_accuracy": float(group["mlp_correct"].mean()),
        "mlp_top5_accuracy": float(group["mlp_top5_correct"].mean()),
        "mlp_macro_f1": float(f1_score(true_indices, mlp_pred_indices, average="macro")),
        "mean_requests": float(group["num_requests"].mean()),
        "median_requests": float(group["num_requests"].median()),
        "cap_hit_count": int((group["num_requests"] >= MAX_REQUEST_CAP).sum()),
        "stop_before_cap_rate": float((group["num_requests"] < MAX_REQUEST_CAP).mean()),
        "mean_final_fused_confidence": float(group["final_fused_confidence"].mean()),
        "mean_final_fused_entropy": float(group["final_fused_entropy"].mean()),
        "mean_final_contradiction_score": float(group["final_contradiction_score"].mean()),
        "mean_final_max_remaining_utility": float(group["final_max_remaining_utility"].mean()),
    }


policy_sweep_summary = pd.DataFrame([
    summarize_policy_group(group)
    for _, group in offline_agent_predictions.groupby("lambda_cost", sort=True)
]).sort_values("lambda_cost")
policy_sweep_summary.to_csv(ARTIFACT_ROOT / "policy_sweep_summary.csv", index=False)
display(policy_sweep_summary)

reference_rows = [{
    "system": "notebook13_selected_stop_49case",
    "accuracy": float(notebook13_metrics["accuracy"]),
    "top3_accuracy": float(notebook13_metrics["top3_accuracy"]),
    "top5_accuracy": float(notebook13_metrics["top5_accuracy"]),
    "macro_f1": float(notebook13_metrics["macro_f1"]),
    "mean_requests": float(notebook13_metrics["mean_requests"]),
    "num_cases": int(notebook13_metrics["num_cases"]),
    "source": str(NOTEBOOK13_RUN / "metrics.json"),
}]
for name, run_dir in [("notebook17_graph_hard_pilot24", NOTEBOOK17_RUN), ("notebook18_graph_advisory_pilot24", NOTEBOOK18_RUN)]:
    metrics_path = run_dir / "metrics.json"
    if metrics_path.exists():
        with metrics_path.open("r", encoding="utf-8") as handle:
            m = json.load(handle)
        reference_rows.append({
            "system": name,
            "accuracy": float(m["accuracy"]),
            "top3_accuracy": float(m.get("top3_accuracy", np.nan)),
            "top5_accuracy": float(m.get("top5_accuracy", np.nan)),
            "macro_f1": float(m.get("macro_f1", np.nan)),
            "mean_requests": float(m["mean_requests"]),
            "num_cases": int(m["num_cases"]),
            "source": str(metrics_path),
        })
for _, row in policy_sweep_summary.iterrows():
    reference_rows.append({
        "system": f"notebook19_bayesian_voi_lambda_{row['lambda_cost']:.2f}",
        "accuracy": float(row["fused_accuracy"]),
        "top3_accuracy": float(row["fused_top3_accuracy"]),
        "top5_accuracy": float(row["fused_top5_accuracy"]),
        "macro_f1": float(row["fused_macro_f1"]),
        "mean_requests": float(row["mean_requests"]),
        "num_cases": int(row["num_cases"]),
        "source": str(ARTIFACT_ROOT / "offline_agent_predictions.csv"),
    })
reference_comparison = pd.DataFrame(reference_rows)
reference_comparison.to_csv(ARTIFACT_ROOT / "reference_comparison.csv", index=False)

# Hard-case audits.
hard_case_ids = ["test:38475", "test:81691", "test:62878", "test:16097", "test:51421", "test:77908"]
hard_case_audits = {}
trace_by_key = {(r["lambda_cost"], r["case_id"]): r for r in all_traces}
for case_id in hard_case_ids:
    case_rows = offline_agent_predictions[offline_agent_predictions["case_id"] == case_id]
    if len(case_rows) == 0:
        continue
    nb13_row = notebook13_predictions[notebook13_predictions["case_id"].astype(str) == case_id]
    hard_case_audits[case_id] = {
        "notebook13": nb13_row.iloc[0].to_dict() if len(nb13_row) else {},
        "notebook19_rows": case_rows.to_dict(orient="records"),
        "traces": {
            str(lambda_cost): trace_by_key.get((float(lambda_cost), case_id), {})
            for lambda_cost in case_rows["lambda_cost"].unique().tolist()
        },
    }
with (ARTIFACT_ROOT / "hard_case_bayes_audits.json").open("w", encoding="utf-8") as handle:
    json.dump(hard_case_audits, handle, indent=2, ensure_ascii=False)

# Promotion decision.
eligible = policy_sweep_summary.copy()
eligible["request_gap_vs_notebook13"] = eligible["mean_requests"] - float(notebook13_metrics["mean_requests"])
best = eligible.sort_values(["fused_correct_count", "mean_requests"], ascending=[False, True]).iloc[0].to_dict()
notebook13_correct_count = int(round(float(notebook13_metrics["accuracy"]) * int(notebook13_metrics["num_cases"])))
fix_counts = {}
for _, row in offline_agent_predictions.groupby("lambda_cost"):
    lam = float(row["lambda_cost"].iloc[0])
    merged = row.merge(notebook13_predictions[["case_id", "correct"]], on="case_id", how="left", suffixes=("_n19", "_n13"))
    fixes = int(((merged["correct"] == False) & (merged["fused_correct"] == True)).sum()) if "correct" in merged else 0
    regressions = int(((merged["correct"] == True) & (merged["fused_correct"] == False)).sum()) if "correct" in merged else 0
    fix_counts[str(lam)] = {"fixes_vs_notebook13": fixes, "regressions_vs_notebook13": regressions}

promote = False
reason = []
if int(best["fused_correct_count"]) >= notebook13_correct_count and float(best["mean_requests"]) <= float(notebook13_metrics["mean_requests"]):
    promote = True
    reason.append("matches_or_beats_notebook13_accuracy_with_no_more_requests")
if int(best["fused_correct_count"]) >= notebook13_correct_count + 1 and float(best["mean_requests"]) <= 9.0:
    promote = True
    reason.append("beats_notebook13_accuracy_with_acceptable_request_count")
for lam, counts in fix_counts.items():
    if int(best["fused_correct_count"]) >= notebook13_correct_count and counts["fixes_vs_notebook13"] >= 2 and counts["regressions_vs_notebook13"] <= 1:
        promote = True
        reason.append("fixes_two_or_more_persistent_cases_without_major_regression")

promotion_decision = {
    "decision": "promote_to_live_notebook20" if promote and not SMOKE_MODE else "do_not_promote_yet" if not SMOKE_MODE else "smoke_run_not_for_promotion",
    "smoke_mode": bool(SMOKE_MODE),
    "reference_notebook13": {
        "correct_count": notebook13_correct_count,
        "num_cases": int(notebook13_metrics["num_cases"]),
        "accuracy": float(notebook13_metrics["accuracy"]),
        "mean_requests": float(notebook13_metrics["mean_requests"]),
    },
    "best_notebook19_policy": best,
    "fix_regression_counts_by_lambda": fix_counts,
    "promotion_reasons": reason,
    "promotion_rule": "promote only if >=43/49 at <=6.59 requests, or >=44/49 at <=9 requests, or same accuracy with >=2 fixes and <=1 regression",
}
with (ARTIFACT_ROOT / "promotion_decision.json").open("w", encoding="utf-8") as handle:
    json.dump(promotion_decision, handle, indent=2)

resolved_run_config = {
    "notebook": "19_bayesian_voi_algorithmic_ledger_offline.ipynb",
    "run_name": RUN_NAME,
    "smoke_mode": bool(SMOKE_MODE),
    "dataset_dir": str(DATASET_DIR),
    "artifact_root": str(ARTIFACT_ROOT),
    "random_seed": RANDOM_SEED,
    "train_likelihood_rows": int(len(raw_train)),
    "calibration_rows": int(len(raw_validate_full)),
    "num_primary_cases": int(len(primary_test_cases)),
    "max_request_cap": int(MAX_REQUEST_CAP),
    "lambda_costs": LAMBDA_COSTS,
    "alpha": ALPHA,
    "log_likelihood_clip": LOG_LIKELIHOOD_CLIP,
    "age_sex_weight": AGE_SEX_WEIGHT,
    "fusion_weights": {"mlp": MLP_FUSION_WEIGHT, "bayes": BAYES_FUSION_WEIGHT},
    "top_bayes_voi_roots": TOP_BAYES_VOI_ROOTS,
    "max_counterfactual_outcomes": MAX_COUNTERFACTUAL_OUTCOMES,
    "stop_config": STOP_CONFIG,
    "utility_weights": UTILITY_WEIGHTS,
    "fairness": "Train split statistics only inside policy; test labels/differentials are used only for evaluation after predictions are generated.",
    "no_api_calls": True,
}
with (ARTIFACT_ROOT / "resolved_run_config.json").open("w", encoding="utf-8") as handle:
    json.dump(resolved_run_config, handle, indent=2)

# Figures.
plt.figure(figsize=(8, 5))
plt.plot(policy_sweep_summary["mean_requests"], policy_sweep_summary["fused_accuracy"], marker="o", label="top-1")
plt.plot(policy_sweep_summary["mean_requests"], policy_sweep_summary["fused_top3_accuracy"], marker="o", label="top-3")
plt.plot(policy_sweep_summary["mean_requests"], policy_sweep_summary["fused_top5_accuracy"], marker="o", label="top-5")
plt.axhline(float(notebook13_metrics["accuracy"]), color="black", linestyle="--", label="Notebook 13 top-1")
plt.axvline(float(notebook13_metrics["mean_requests"]), color="gray", linestyle=":", label="Notebook 13 requests")
plt.xlabel("Mean requested evidence fields")
plt.ylabel("Accuracy")
plt.title("Bayesian VOI Performance vs Evidence Requests")
plt.legend()
plt.tight_layout()
plt.savefig(FIGURE_DIR / "accuracy_topk_vs_mean_requests.png", dpi=160)
plt.close()

plt.figure(figsize=(8, 5))
plt.plot(policy_sweep_summary["lambda_cost"], policy_sweep_summary["fused_accuracy"], marker="o", label="accuracy")
plt.plot(policy_sweep_summary["lambda_cost"], policy_sweep_summary["mean_requests"], marker="s", label="mean requests")
plt.xlabel("Lambda evidence cost")
plt.title("Lambda Cost Sweep")
plt.legend()
plt.tight_layout()
plt.savefig(FIGURE_DIR / "utility_vs_lambda.png", dpi=160)
plt.close()

trace_turn_rows = []
for record in all_traces:
    for turn in record["trace"]:
        trace_turn_rows.append({
            "lambda_cost": record["lambda_cost"],
            "case_id": record["case_id"],
            "turn_index": turn["turn_index"],
            "fused_entropy": turn["fused_summary"]["entropy"],
            "max_remaining_utility": turn["max_remaining_utility"],
            "contradiction_score": turn["contradiction_score"],
            "decision": turn["decision"],
        })
turn_trace_df = pd.DataFrame(trace_turn_rows)
if len(turn_trace_df):
    turn_trace_df.groupby("turn_index")["fused_entropy"].mean().plot(marker="o", figsize=(8, 5), title="Mean Fused Posterior Entropy Over Turns")
    plt.xlabel("Turn")
    plt.ylabel("Normalized entropy")
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / "posterior_entropy_over_turns.png", dpi=160)
    plt.close()

    turn_trace_df.groupby("turn_index")["max_remaining_utility"].mean().plot(marker="o", figsize=(8, 5), title="Mean Max Remaining VOI Utility Over Turns")
    plt.xlabel("Turn")
    plt.ylabel("Max remaining utility")
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / "max_remaining_voi_over_turns.png", dpi=160)
    plt.close()

plt.figure(figsize=(8, 5))
offline_agent_predictions.boxplot(column="final_contradiction_score", by="fused_correct", ax=plt.gca())
plt.title("Contradiction Score By Correctness")
plt.suptitle("")
plt.xlabel("Fused final correct")
plt.ylabel("Final contradiction score")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "contradiction_score_by_correctness.png", dpi=160)
plt.close()

head_summary = []
for _, group in offline_agent_predictions.groupby("lambda_cost"):
    head_summary.extend([
        {"lambda_cost": group["lambda_cost"].iloc[0], "head": "Bayes", "accuracy": group["bayes_correct"].mean()},
        {"lambda_cost": group["lambda_cost"].iloc[0], "head": "MLP", "accuracy": group["mlp_correct"].mean()},
        {"lambda_cost": group["lambda_cost"].iloc[0], "head": "Fused", "accuracy": group["fused_correct"].mean()},
    ])
head_df = pd.DataFrame(head_summary)
if len(head_df):
    pivot = head_df.pivot(index="lambda_cost", columns="head", values="accuracy")
    pivot.plot(kind="bar", figsize=(8, 5), title="Bayes vs MLP vs Fused Final Heads")
    plt.ylabel("Accuracy")
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / "final_head_accuracy.png", dpi=160)
    plt.close()

requested_counter = Counter()
for roots_json in offline_agent_predictions["requested_roots"]:
    requested_counter.update(json.loads(roots_json))
requested_frequency = pd.DataFrame([
    {"root_evidence_id": root, "question_en": schema.question_text.get(root, root), "count": count}
    for root, count in requested_counter.most_common()
])
requested_frequency.to_csv(ARTIFACT_ROOT / "requested_evidence_frequency.csv", index=False)
if len(requested_frequency):
    top_freq = requested_frequency.head(20).iloc[::-1]
    plt.figure(figsize=(8, 7))
    plt.barh(top_freq["root_evidence_id"], top_freq["count"])
    plt.xlabel("Request count")
    plt.title("Most Requested Evidence Roots")
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / "requested_evidence_frequency.png", dpi=160)
    plt.close()

# Hard-case timeline figure.
hard_trace_rows = turn_trace_df[turn_trace_df["case_id"].isin(hard_case_ids)] if len(turn_trace_df) else pd.DataFrame()
if len(hard_trace_rows):
    plt.figure(figsize=(10, 6))
    for case_id, group in hard_trace_rows.groupby("case_id"):
        first_lambda = sorted(group["lambda_cost"].unique())[0]
        g = group[group["lambda_cost"] == first_lambda]
        plt.plot(g["turn_index"], g["fused_entropy"], marker="o", label=case_id)
    plt.xlabel("Turn")
    plt.ylabel("Fused entropy")
    plt.title("Hard-Case Posterior Entropy Timeline")
    plt.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / "hard_case_entropy_timelines.png", dpi=160)
    plt.close()

print("Promotion decision:")
print(json.dumps(promotion_decision, indent=2)[:4000])
print("Saved artifacts to", ARTIFACT_ROOT)

,lambda_cost,num_cases,fused_correct_count,fused_accuracy,fused_top3_accuracy,fused_top5_accuracy,fused_macro_f1,bayes_accuracy,bayes_top5_accuracy,bayes_macro_f1,...,mlp_top5_accuracy,mlp_macro_f1,mean_requests,median_requests,cap_hit_count,stop_before_cap_rate,mean_final_fused_confidence,mean_final_fused_entropy,mean_final_contradiction_score,mean_final_max_remaining_utility
0,0.00,49,33,0.673469,0.836735,0.877551,0.620408,0.714286,0.816327,0.648980,...,0.857143,0.620408,22.367347,24.0,28,0.428571,0.984982,0.019550,0.872365,0.001550
1,0.02,49,26,0.530612,0.734694,0.836735,0.460544,0.489796,0.693878,0.421202,...,0.795918,0.460544,7.979592,7.0,4,0.918367,0.956118,0.062152,0.358503,-0.006707
2,0.05,49,26,0.530612,0.632653,0.775510,0.447328,0.408163,0.653061,0.335714,...,0.755102,0.447328,5.653061,5.0,1,0.979592,0.888459,0.133384,0.099206,-0.016371
3,0.10,49,25,0.510204,0.653061,0.775510,0.423518,0.326531,0.591837,0.253711,...,0.795918,0.423518,4.428571,3.0,1,0.979592,0.807473,0.211819,0.041737,-0.039698
4,0.15,49,24,0.489796,0.632653,0.775510,0.396307,0.326531,0.591837,0.250309,...,0.795918,0.396307,4.326531,3.0,1,0.979592,0.800536,0.215436,0.041737,-0.087187


Promotion decision:
{
  "decision": "do_not_promote_yet",
  "smoke_mode": false,
  "reference_notebook13": {
    "correct_count": 43,
    "num_cases": 49,
    "accuracy": 0.8775510204081632,
    "mean_requests": 6.591836734693878
  },
  "best_notebook19_policy": {
    "lambda_cost": 0.0,
    "num_cases": 49.0,
    "fused_correct_count": 33.0,
    "fused_accuracy": 0.673469387755102,
    "fused_top3_accuracy": 0.8367346938775511,
    "fused_top5_accuracy": 0.8775510204081632,
    "fused_macro_f1": 0.6204081632653061,
    "bayes_accuracy": 0.7142857142857143,
    "bayes_top5_accuracy": 0.8163265306122449,
    "bayes_macro_f1": 0.6489795918367347,
    "mlp_accuracy": 0.673469387755102,
    "mlp_top5_accuracy": 0.8571428571428571,
    "mlp_macro_f1": 0.6204081632653061,
    "mean_requests": 22.367346938775512,
    "median_requests": 24.0,
    "cap_hit_count": 28.0,
    "stop_before_cap_rate": 0.42857142857142855,
    "mean_final_fused_confidence": 0.9849822718707753,
    "mean_final_fused_

## 9. Summary

Use the promotion decision and policy sweep table to decide whether this Bayesian VOI ledger justifies a live Notebook 20. Smoke-mode metrics are not scientific; they only validate execution and artifact contracts.

In [10]:
print("Run directory:", ARTIFACT_ROOT)
print("Smoke mode   :", SMOKE_MODE)
print("Best fused policy:")
display(policy_sweep_summary.sort_values(["fused_correct_count", "mean_requests"], ascending=[False, True]).head(5))
print("Reference comparison:")
display(reference_comparison)

Run directory: /Users/bilalawan/claw/assignments/baseline_model/artifacts/bayesian_voi_ledger/bayesian_voi_offline_notebook13_49case_v1
Smoke mode   : False
Best fused policy:


,lambda_cost,num_cases,fused_correct_count,fused_accuracy,fused_top3_accuracy,fused_top5_accuracy,fused_macro_f1,bayes_accuracy,bayes_top5_accuracy,bayes_macro_f1,...,mlp_top5_accuracy,mlp_macro_f1,mean_requests,median_requests,cap_hit_count,stop_before_cap_rate,mean_final_fused_confidence,mean_final_fused_entropy,mean_final_contradiction_score,mean_final_max_remaining_utility
0,0.00,49,33,0.673469,0.836735,0.877551,0.620408,0.714286,0.816327,0.648980,...,0.857143,0.620408,22.367347,24.0,28,0.428571,0.984982,0.019550,0.872365,0.001550
2,0.05,49,26,0.530612,0.632653,0.775510,0.447328,0.408163,0.653061,0.335714,...,0.755102,0.447328,5.653061,5.0,1,0.979592,0.888459,0.133384,0.099206,-0.016371
1,0.02,49,26,0.530612,0.734694,0.836735,0.460544,0.489796,0.693878,0.421202,...,0.795918,0.460544,7.979592,7.0,4,0.918367,0.956118,0.062152,0.358503,-0.006707
3,0.10,49,25,0.510204,0.653061,0.775510,0.423518,0.326531,0.591837,0.253711,...,0.795918,0.423518,4.428571,3.0,1,0.979592,0.807473,0.211819,0.041737,-0.039698
4,0.15,49,24,0.489796,0.632653,0.775510,0.396307,0.326531,0.591837,0.250309,...,0.795918,0.396307,4.326531,3.0,1,0.979592,0.800536,0.215436,0.041737,-0.087187


Reference comparison:


,system,accuracy,top3_accuracy,top5_accuracy,macro_f1,mean_requests,num_cases,source
0,notebook13_selected_stop_49case,0.877551,0.918367,0.938776,0.844898,6.591837,49,/Users/bilalawan/claw/assignments/baseline_model/artifacts/sequential_hybrid_mlp_feedback/selected_stop_live_confirmation_49case_v1/metrics.json
1,notebook17_graph_hard_pilot24,0.833333,0.833333,0.875000,0.743590,6.208333,24,/Users/bilalawan/claw/assignments/baseline_model/artifacts/graph_algorithmic_ledger/live_medkgi_graph_shortlist_pilot24_v1/metrics.json
2,notebook18_graph_advisory_pilot24,0.875000,0.875000,0.916667,0.794872,7.666667,24,/Users/bilalawan/claw/assignments/baseline_model/artifacts/graph_algorithmic_ledger/live_graph_advisory_hybrid_shortlist_pilot24_v1/metrics.json
3,notebook19_bayesian_voi_lambda_0.00,0.673469,0.836735,0.877551,0.620408,22.367347,49,/Users/bilalawan/claw/assignments/baseline_model/artifacts/bayesian_voi_ledger/bayesian_voi_offline_notebook13_49case_v1/offline_agent_predictions.csv
4,notebook19_bayesian_voi_lambda_0.02,0.530612,0.734694,0.836735,0.460544,7.979592,49,/Users/bilalawan/claw/assignments/baseline_model/artifacts/bayesian_voi_ledger/bayesian_voi_offline_notebook13_49case_v1/offline_agent_predictions.csv
5,notebook19_bayesian_voi_lambda_0.05,0.530612,0.632653,0.775510,0.447328,5.653061,49,/Users/bilalawan/claw/assignments/baseline_model/artifacts/bayesian_voi_ledger/bayesian_voi_offline_notebook13_49case_v1/offline_agent_predictions.csv
6,notebook19_bayesian_voi_lambda_0.10,0.510204,0.653061,0.775510,0.423518,4.428571,49,/Users/bilalawan/claw/assignments/baseline_model/artifacts/bayesian_voi_ledger/bayesian_voi_offline_notebook13_49case_v1/offline_agent_predictions.csv
7,notebook19_bayesian_voi_lambda_0.15,0.489796,0.632653,0.775510,0.396307,4.326531,49,/Users/bilalawan/claw/assignments/baseline_model/artifacts/bayesian_voi_ledger/bayesian_voi_offline_notebook13_49case_v1/offline_agent_predictions.csv
